In [ ]:
# Colab bootstrap: install dependencies, mount Google Drive, and create experiment folders
from pathlib import Path
from google.colab import drive

# --- Install dependencies ---
!pip install -q --upgrade pip
!pip install -q openml pmlb keel-ds xgboost scikit-learn matplotlib seaborn weightwatcher

# Install xgboost2ww from source (requested flow)
!rm -rf /content/repo_xgboost2ww
!git clone https://github.com/CalculatedContent/xgboost2ww.git /content/repo_xgboost2ww
!pip install -q -e /content/repo_xgboost2ww

# Install xgbwwdata from source (same flow used in catalog notebooks)
!rm -rf /content/repo_xgbwwdata
!git clone https://github.com/CalculatedContent/xgbwwdata.git /content/repo_xgbwwdata
%run /content/repo_xgbwwdata/scripts/colab_install.py --repo /content/repo_xgbwwdata

# --- Mount Drive and prepare experiment directory ---
drive.mount('/content/drive')

EXPERIMENT_ROOT = Path('/content/drive/MyDrive/xgbwwdata/experiment_checkpoints')
EXPERIMENT_NAME = 'random100_longrun_alpha_tracking'
EXPERIMENT_DIR = EXPERIMENT_ROOT / EXPERIMENT_NAME
PLOTS_DIR = EXPERIMENT_DIR / 'plots_top10'

EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print('Experiment directory:', EXPERIMENT_DIR)

# XGBWW Catalog Random100 — Long-Run Alpha Tracking

This notebook copies the Random100 catalog workflow, but runs each selected model for a long time and checkpoints:
- alpha for **W1, W2, W7, W8, W9, W10**
- train / validation / test accuracy
- all tracked values every N boosting steps

All outputs are written to Google Drive checkpoint files.

In [ ]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import xgboost as xgb
import weightwatcher as ww

from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from xgbwwdata import Filters, load_dataset
from xgboost2ww import convert

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

# ===== USER CONFIG =====
CATALOG_CSV = Path('/content/drive/MyDrive/xgbwwdata/catalog_checkpoint/dataset_catalog.csv')
RANDOM_SEED = 42
N_MODELS = 100
TEST_SIZE = 0.20
VAL_SIZE_FROM_TRAIN = 0.20

TOTAL_BOOST_ROUNDS = 3000      # Long run to allow overfitting
CHECKPOINT_EVERY = 100         # Track metrics every N rounds
EARLY_STOPPING_ROUNDS = None   # Keep None to intentionally allow overfit

TARGET_WS = ['W1', 'W2', 'W7', 'W8', 'W9', 'W10']

RESULTS_CSV = EXPERIMENT_DIR / 'model_level_results.csv'
TIMESERIES_CSV = EXPERIMENT_DIR / 'alpha_accuracy_timeseries.csv'
ERRORS_CSV = EXPERIMENT_DIR / 'errors.csv'
CONFIG_JSON = EXPERIMENT_DIR / 'experiment_config.json'

In [ ]:
if not CATALOG_CSV.exists():
    raise FileNotFoundError(f'Catalog not found: {CATALOG_CSV}')

catalog = pd.read_csv(CATALOG_CSV)
required_cols = {'dataset_uid', 'source', 'task_type'}
missing = required_cols - set(catalog.columns)
if missing:
    raise ValueError(f'Missing required catalog columns: {missing}')

cls_catalog = catalog[catalog['task_type'].astype(str).str.contains('classification', case=False, na=False)].copy()
if cls_catalog.empty:
    raise ValueError('No classification datasets found in catalog.')

n_pick = min(N_MODELS, len(cls_catalog))
selected = cls_catalog.sample(n=n_pick, random_state=RANDOM_SEED).reset_index(drop=True)
selected_path = EXPERIMENT_DIR / 'selected_datasets.csv'
selected.to_csv(selected_path, index=False)

print('Catalog rows:', len(catalog))
print('Classification rows:', len(cls_catalog))
print('Selected models:', len(selected))
selected[['source', 'dataset_uid', 'name']].head()

In [ ]:
filters = Filters(min_rows=200, max_rows=60000, max_features=50000, max_dense_elements=int(2e8))


def detect_compute_params():
    probe_X = np.array([[0.0], [1.0], [2.0], [3.0]], dtype=np.float32)
    probe_y = np.array([0, 0, 1, 1], dtype=np.float32)
    dprobe = xgb.DMatrix(probe_X, label=probe_y)
    gpu_params = {'tree_method': 'hist', 'device': 'cuda'}
    cpu_params = {'tree_method': 'hist'}
    try:
        xgb.train({'objective': 'binary:logistic', 'eval_metric': 'logloss', **gpu_params}, dprobe, num_boost_round=1, verbose_eval=False)
        return gpu_params, 'gpu'
    except Exception:
        return cpu_params, 'cpu'


def infer_w_label(row):
    for c in ['name', 'layer_name', 'longname', 'matrix_name']:
        if c in row and pd.notna(row[c]):
            txt = str(row[c]).upper()
            for w in TARGET_WS:
                if w in txt:
                    return w
    if 'layer_id' in row and pd.notna(row['layer_id']):
        return f"W{int(row['layer_id'])}"
    return None


def collect_alpha_snapshot(booster, wlist=TARGET_WS):
    details = convert(booster)
    watcher = ww.WeightWatcher(model=details)
    ww_df = watcher.analyze(plot=False, randomize=False)
    ww_df = ww_df.copy()

    labels = [infer_w_label(r) for _, r in ww_df.iterrows()]
    ww_df['w_label'] = labels
    alpha_map = {w: np.nan for w in wlist}
    trap_map = {w: np.nan for w in wlist}

    for w in wlist:
        rows = ww_df[ww_df['w_label'] == w]
        if not rows.empty:
            if 'alpha' in rows.columns:
                alpha_map[w] = float(rows['alpha'].astype(float).iloc[-1])
            if 'num_traps' in rows.columns:
                trap_map[w] = float(rows['num_traps'].astype(float).iloc[-1])

    total_traps = np.nan
    if 'num_traps' in ww_df.columns:
        total_traps = float(pd.to_numeric(ww_df['num_traps'], errors='coerce').fillna(0).sum())

    return alpha_map, trap_map, total_traps


def prepare_labels(y_raw):
    y_series = pd.Series(y_raw)
    if y_series.dtype.kind in {'i', 'u', 'f'}:
        y_vals = y_series.astype(int).values
    else:
        enc = LabelEncoder()
        y_vals = enc.fit_transform(y_series.astype(str).values)
    return y_vals


COMPUTE_PARAMS, BACKEND = detect_compute_params()
print('XGBoost backend:', BACKEND, '| params:', COMPUTE_PARAMS)

In [ ]:
model_rows = []
timeseries_rows = []
error_rows = []

for idx, row in selected.iterrows():
    dataset_uid = str(row['dataset_uid'])
    source = str(row['source'])
    dataset_name = str(row.get('name', dataset_uid))
    print(f"[{idx+1}/{len(selected)}] {dataset_uid} ({source})")

    try:
        ds = load_dataset(dataset_uid, source=source, filters=filters)
        X = ds.X
        y = prepare_labels(ds.y)

        stratify_arg = y if len(np.unique(y)) > 1 else None
        X_train_all, X_test, y_train_all, y_test = train_test_split(
            X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=stratify_arg
        )

        stratify_train = y_train_all if len(np.unique(y_train_all)) > 1 else None
        X_train, X_val, y_train, y_val = train_test_split(
            X_train_all, y_train_all, test_size=VAL_SIZE_FROM_TRAIN,
            random_state=RANDOM_SEED, stratify=stratify_train
        )

        n_classes = int(len(np.unique(y)))
        if n_classes <= 2:
            objective = 'binary:logistic'
            eval_metric = 'logloss'
        else:
            objective = 'multi:softprob'
            eval_metric = 'mlogloss'

        params = {
            'objective': objective,
            'eval_metric': eval_metric,
            'eta': 0.03,
            'max_depth': 10,
            'min_child_weight': 1,
            'subsample': 1.0,
            'colsample_bytree': 1.0,
            'reg_lambda': 0.0,
            'reg_alpha': 0.0,
            'seed': RANDOM_SEED,
            **COMPUTE_PARAMS,
        }
        if n_classes > 2:
            params['num_class'] = n_classes

        dtrain = xgb.DMatrix(X_train, label=y_train)
        dval = xgb.DMatrix(X_val, label=y_val)
        dtest = xgb.DMatrix(X_test, label=y_test)

        booster = None
        for step in range(CHECKPOINT_EVERY, TOTAL_BOOST_ROUNDS + 1, CHECKPOINT_EVERY):
            rounds_this_chunk = CHECKPOINT_EVERY if booster is None else CHECKPOINT_EVERY
            booster = xgb.train(
                params=params,
                dtrain=dtrain,
                num_boost_round=rounds_this_chunk,
                evals=[(dtrain, 'train'), (dval, 'val')],
                xgb_model=booster,
                early_stopping_rounds=EARLY_STOPPING_ROUNDS,
                verbose_eval=False,
            )

            if n_classes <= 2:
                train_pred = (booster.predict(dtrain) >= 0.5).astype(int)
                val_pred = (booster.predict(dval) >= 0.5).astype(int)
                test_pred = (booster.predict(dtest) >= 0.5).astype(int)
            else:
                train_pred = np.argmax(booster.predict(dtrain), axis=1)
                val_pred = np.argmax(booster.predict(dval), axis=1)
                test_pred = np.argmax(booster.predict(dtest), axis=1)

            train_acc = float(accuracy_score(y_train, train_pred))
            val_acc = float(accuracy_score(y_val, val_pred))
            test_acc = float(accuracy_score(y_test, test_pred))

            alpha_map, trap_map, total_traps = collect_alpha_snapshot(booster, TARGET_WS)
            trow = {
                'dataset_uid': dataset_uid,
                'source': source,
                'name': dataset_name,
                'n_rounds': int(step),
                'train_accuracy': train_acc,
                'val_accuracy': val_acc,
                'test_accuracy': test_acc,
            }
            for w in TARGET_WS:
                trow[f'alpha_{w}'] = alpha_map.get(w, np.nan)
                trow[f'num_traps_{w}'] = trap_map.get(w, np.nan)
            trow['num_traps_total'] = total_traps
            timeseries_rows.append(trow)

            # Frequent checkpoint flush to Drive
            pd.DataFrame(timeseries_rows).to_csv(TIMESERIES_CSV, index=False)

        final = pd.DataFrame(timeseries_rows)
        final = final[final['dataset_uid'] == dataset_uid].sort_values('n_rounds').iloc[-1]
        model_rows.append({
            'dataset_uid': dataset_uid,
            'source': source,
            'name': dataset_name,
            'final_rounds': int(final['n_rounds']),
            'train_accuracy': float(final['train_accuracy']),
            'val_accuracy': float(final['val_accuracy']),
            'test_accuracy': float(final['test_accuracy']),
            **{f'alpha_{w}': float(final[f'alpha_{w}']) if pd.notna(final[f'alpha_{w}']) else np.nan for w in TARGET_WS},
            **{f'num_traps_{w}': float(final[f'num_traps_{w}']) if pd.notna(final[f'num_traps_{w}']) else np.nan for w in TARGET_WS},
            'num_traps_total': float(final['num_traps_total']) if pd.notna(final['num_traps_total']) else np.nan,
            'status': 'completed',
        })

        pd.DataFrame(model_rows).to_csv(RESULTS_CSV, index=False)

    except Exception as e:
        error_rows.append({'dataset_uid': dataset_uid, 'source': source, 'error': str(e)})
        pd.DataFrame(error_rows).to_csv(ERRORS_CSV, index=False)
        print('  -> FAILED:', e)

# Save run config and final outputs
config = {
    'random_seed': RANDOM_SEED,
    'n_models_requested': N_MODELS,
    'n_models_selected': int(len(selected)),
    'total_boost_rounds': TOTAL_BOOST_ROUNDS,
    'checkpoint_every': CHECKPOINT_EVERY,
    'target_ws': TARGET_WS,
    'backend': BACKEND,
}
with open(CONFIG_JSON, 'w') as f:
    json.dump(config, f, indent=2)

pd.DataFrame(model_rows).to_csv(RESULTS_CSV, index=False)
pd.DataFrame(timeseries_rows).to_csv(TIMESERIES_CSV, index=False)
pd.DataFrame(error_rows).to_csv(ERRORS_CSV, index=False)

print('Done. Files saved to:', EXPERIMENT_DIR)
print('RESULTS_CSV:', RESULTS_CSV)
print('TIMESERIES_CSV:', TIMESERIES_CSV)
print('ERRORS_CSV:', ERRORS_CSV)

In [ ]:
# Pick best-performing models, excluding too-simple models, then plot top-10 alpha trajectories
if not RESULTS_CSV.exists() or not TIMESERIES_CSV.exists():
    raise FileNotFoundError('Run training first so checkpoint files are available.')

results_df = pd.read_csv(RESULTS_CSV)
ts_df = pd.read_csv(TIMESERIES_CSV)

if results_df.empty:
    raise ValueError('No successful model results found.')

# Quality filter requested by user:
# - reject correlation traps (num_traps_total > 0)
# - reject weak W7 alpha (alpha_W7 < 2)
# - reject perfectly-fit models (train==1 and test==1)
results_df['num_traps_total'] = pd.to_numeric(results_df.get('num_traps_total'), errors='coerce').fillna(0)
results_df['alpha_W7'] = pd.to_numeric(results_df.get('alpha_W7'), errors='coerce')
results_df['train_accuracy'] = pd.to_numeric(results_df.get('train_accuracy'), errors='coerce')
results_df['test_accuracy'] = pd.to_numeric(results_df.get('test_accuracy'), errors='coerce')

filtered_df = results_df[
    (results_df['num_traps_total'] <= 0) &
    (results_df['alpha_W7'] >= 2.0) &
    ~((results_df['train_accuracy'] >= 0.999999) & (results_df['test_accuracy'] >= 0.999999))
].copy()

print('Total completed models:', len(results_df))
print('Models after quality filter:', len(filtered_df))

rank_source = filtered_df if len(filtered_df) > 0 else results_df
if len(filtered_df) == 0:
    print('Warning: no models passed the strict filter. Falling back to all completed models.')

top10 = rank_source.sort_values('test_accuracy', ascending=False).head(10).copy()
print(top10[['dataset_uid', 'source', 'name', 'test_accuracy', 'alpha_W7', 'num_traps_total']])

top10.to_csv(EXPERIMENT_DIR / 'top10_models.csv', index=False)
filtered_df.to_csv(EXPERIMENT_DIR / 'filtered_candidate_models.csv', index=False)

for rank, (_, mrow) in enumerate(top10.iterrows(), start=1):
    uid = str(mrow['dataset_uid'])
    m_ts = ts_df[ts_df['dataset_uid'].astype(str) == uid].sort_values('n_rounds')
    if m_ts.empty:
        continue

    plt.figure(figsize=(10, 5))
    for w in TARGET_WS:
        col = f'alpha_{w}'
        if col in m_ts.columns:
            plt.plot(m_ts['n_rounds'], m_ts[col], marker='o', linewidth=2, label=w)

    plt.title(f"Top-{rank} | {mrow['name']} ({uid}) | test_acc={mrow['test_accuracy']:.4f}")
    plt.xlabel('Boosting rounds (N steps)')
    plt.ylabel('WeightWatcher alpha')
    plt.legend(ncol=3)
    plt.tight_layout()

    out_path = PLOTS_DIR / f'top{rank:02d}_{uid}_alpha_trajectory.png'
    plt.savefig(out_path, dpi=150)
    plt.show()

print('Saved top-10 plots to:', PLOTS_DIR)
